# 1. Cleaning and Imputation

In [16]:
import pandas as pd
import numpy as np
import re

## 1. Loading Data and initialization of *Silver Layer*

In [26]:
#path to csv file
file_path = r'C:\Users\Maciek\Desktop\netflixdb\databases\movies.csv'

#Loading raw data
df_bronze = pd.read_csv(file_path, sep = ',')

#Verification
print(f"Number of columns in df: {df_bronze.shape[1]}")
print("First 5 rows: ")
print(df_bronze.head())

#Coping bronze layer into silver layer (df_silver) for secure data transformations
df_silver = df_bronze.copy()



Number of columns in df: 18
First 5 rows: 
     movie_id            title     content_type genre_primary genre_secondary  \
0  movie_0001    Dragon Legend  Stand-up Comedy       History        Thriller   
1  movie_0002    Storm Warrior  Stand-up Comedy        Sci-Fi             NaN   
2  movie_0003      Fire Family            Movie         Drama             NaN   
3  movie_0004     Our Princess      Documentary        Sci-Fi             NaN   
4  movie_0005  Warrior Mission      Documentary         Sport         Mystery   

   release_year  duration_minutes rating  language country_of_origin  \
0          2014              35.0   TV-Y    French             Japan   
1          2017              37.0     PG  Japanese               USA   
2          2003             142.0  TV-MA   English               USA   
3          2011             131.0  NC-17  Japanese               USA   
4          2015              91.0   TV-G   English               USA   

   imdb_rating  production_budget  bo

In [27]:
df_duplicates = df_silver[df_silver['movie_id'].duplicated(keep=False)]
print(df_duplicates[['movie_id', 'title', 'added_to_platform']].sort_values('movie_id'))

        movie_id            title added_to_platform
33    movie_0034  Mission Kingdom        2023-10-22
1013  movie_0034  Mission Kingdom        2023-10-22
34    movie_0035    Princess Love        2021-03-01
1034  movie_0035    Princess Love        2021-03-01
42    movie_0043       City Queen        2022-12-12
...          ...              ...               ...
1021  movie_0964   Journey Empire        2022-11-22
979   movie_0980       War Battle        2024-03-11
1020  movie_0980       War Battle        2024-03-11
1003  movie_0996   Secret Mission        2021-05-28
995   movie_0996   Secret Mission        2021-05-28

[80 rows x 3 columns]


In [28]:
duplicate_count = df_silver['movie_id'].duplicated().sum()
print(f"Liczba duplikatów movie_id: {duplicate_count}")

Liczba duplikatów movie_id: 40


In [29]:
df_silver_clean = df_silver.drop_duplicates(keep='first')

rows_removed = len(df_silver) - len(df_silver_clean)

print(rows_removed)

df_silver = df_silver_clean

40


## 2. Analyses and Exploration Nulls

In [30]:
# Checking number of Nulls (NaN) in each column, to plan imputation
print(df_silver.isnull().sum())
print(df_silver.describe())

movie_id                 0
title                    0
content_type             0
genre_primary            0
genre_secondary        643
release_year             0
duration_minutes         0
rating                   0
language                 0
country_of_origin        0
imdb_rating            144
production_budget      647
box_office_revenue     678
number_of_seasons      725
number_of_episodes     695
is_netflix_original      0
added_to_platform        0
content_warning          0
dtype: int64
       release_year  duration_minutes  imdb_rating  production_budget  \
count   1000.000000       1000.000000   856.000000       3.530000e+02   
mean    2006.446000         89.845000     6.281425       1.119697e+07   
std       11.319506         70.319569     1.801823       2.426292e+07   
min     1953.000000          0.000000     0.500000       6.837300e+04   
25%     1998.000000         51.000000     5.300000       1.442937e+06   
50%     2006.000000         82.000000     6.400000       3.7784

In [31]:
#Showing groups in 'genre_secondary'
print(df_bronze.groupby('genre_secondary')['movie_id'].count())

genre_secondary
Action         11
Adventure      25
Animation      17
Biography      20
Comedy         14
Crime          20
Documentary    14
Drama          28
Family         22
Fantasy        20
History        19
Horror         12
Music          14
Mystery        20
Romance        17
Sci-Fi         23
Sport          15
Thriller       23
War            19
Western        20
Name: movie_id, dtype: int64


## 3. Cleaning and Imputation of Categorical Data

In [32]:
# 'genre_secondary' manipulations

#Standarization: Converting empty strings and white signs to NaN
df_silver['genre_secondary'] = df_silver['genre_secondary'].str.strip().replace('',np.nan)

#Handling Non-standard strings 'NULL' as NaN
df_silver['genre_secondary'] = df_silver['genre_secondary'].replace('NULL', np.nan)

#Imputaion: Filling NaN with 'unknown' value. Making business lvl understanding of data
df_silver['genre_secondary'] = df_silver['genre_secondary'].fillna('unknown')

#Verification check
print(df_silver.isnull().sum())

movie_id                 0
title                    0
content_type             0
genre_primary            0
genre_secondary          0
release_year             0
duration_minutes         0
rating                   0
language                 0
country_of_origin        0
imdb_rating            144
production_budget      647
box_office_revenue     678
number_of_seasons      725
number_of_episodes     695
is_netflix_original      0
added_to_platform        0
content_warning          0
dtype: int64


In [33]:
# handling blank spaces in 'title','rating','language','country_of_origin' columns

columns_to_strip = ['title', 'rating', 'language', 'country_of_origin']
for column in columns_to_strip:
    df_silver[column] = df_silver[column].str.strip()

## 4. IMDb Rating Imputation

In [34]:
# Calculate the median from available data. Median is robust to outliers.
imdb_median = df_silver['imdb_rating'].median()

# Feature Engineering: Create a binary flag (1/0) indicating where the original value was missing.
# This is crucial for ML models to capture the predictive power of missing data itself.
df_silver['is_imdb_rating_missing'] = df_silver['imdb_rating'].isna().astype(int)

# Imputation: Fill NaN with the calculated median to preserve the column's statistical distribution.
df_silver['imdb_rating'] = df_silver['imdb_rating'].fillna(imdb_median)
    
#Verification
print(df_silver.isnull().sum())

movie_id                    0
title                       0
content_type                0
genre_primary               0
genre_secondary             0
release_year                0
duration_minutes            0
rating                      0
language                    0
country_of_origin           0
imdb_rating                 0
production_budget         647
box_office_revenue        678
number_of_seasons         725
number_of_episodes        695
is_netflix_original         0
added_to_platform           0
content_warning             0
is_imdb_rating_missing      0
dtype: int64


## 5. Financial Imputation

In [35]:
#Pattern to extract monetary values (e.g., $1,234,567)
pattern_to_clean = r'[$,\s]'

#Copy of the original column for testing purposes
df_test = df_silver['production_budget'].astype(str).copy()

#Replacing characters matching the pattern with an empty string
df_cleaned_data = df_test.str.replace(pattern_to_clean, '', regex = True)

rows_to_clean = df_test.str.len() != df_cleaned_data.str.len()

print(f"Number of rows to clean in 'production_budget': {rows_to_clean.sum()}")

Number of rows to clean in 'production_budget': 0


In [36]:
#Pattern to extract monetary values (e.g., $1,234,567)
pattern_to_clean = r'[$,\s]'

#Copy of the original column for testing purposes
df_test = df_silver['box_office_revenue'].astype(str).copy()

#Replacing characters matching the pattern with an empty string
df_cleaned_data = df_test.str.replace(pattern_to_clean, '', regex = True)

rows_to_clean = df_test.str.len() != df_cleaned_data.str.len()

print(f"Number of rows to clean in 'box_office_revenue': {rows_to_clean.sum()}")

Number of rows to clean in 'box_office_revenue': 0


In [37]:
#Analysing column schema
print(df_silver['production_budget'])

# Flagging: Create the missingness indicator flag (1/0) before imputation.
df_silver['is_production_budget_missing'] = df_silver['production_budget'].isna().astype(int)

# Imputation: Fill NaN with zero (0). For financial data, missing usually means zero or undisclosed, not the average budget.
df_silver['production_budget'] = df_silver['production_budget'].fillna(0)

# Converting 'production_budget' to float with 4 decimal places
df_silver['production_budget'] = df_silver['production_budget'].round(4).astype(float)

0             NaN
1             NaN
2       2114120.0
3             NaN
4             NaN
          ...    
995           NaN
996           NaN
997           NaN
998    49649970.0
999           NaN
Name: production_budget, Length: 1000, dtype: float64


In [38]:
#Creating binary column to indentify records w/o values
df_silver['is_box_office_revenue_missing'] = df_silver['box_office_revenue'].isna().astype(int)

#Replacing NaN with 0
df_silver['box_office_revenue'] = df_silver['box_office_revenue'].fillna(0)

# Converting 'box_office_revenue' to float with 4 decimal places
df_silver['box_office_revenue'] = df_silver['box_office_revenue'].round(4).astype(float)

#Verification
print(df_silver['box_office_revenue'].isnull().sum())

0


## 6. Series Imputation

In [39]:
series_col = ['number_of_episodes','number_of_seasons']

for col in series_col:
    # Imputation: Fill NaN with 1. This is semantically correct for films/stand-up specials.
    df_silver[col] = df_silver[col].fillna(1)

    # Type Conversion: Ensure the column is an integer type.
    df_silver[col] = df_silver[col].astype(int)

print("\nLiczba NaN po imputacji:")
print(df_silver[series_col].isna().sum()) 
print("\nTypy kolumn po konwersji:")
print(df_silver[series_col].dtypes)


Liczba NaN po imputacji:
number_of_episodes    0
number_of_seasons     0
dtype: int64

Typy kolumn po konwersji:
number_of_episodes    int64
number_of_seasons     int64
dtype: object


# 2. Convertion and Formating

In [40]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   movie_id                       1000 non-null   object 
 1   title                          1000 non-null   object 
 2   content_type                   1000 non-null   object 
 3   genre_primary                  1000 non-null   object 
 4   genre_secondary                1000 non-null   object 
 5   release_year                   1000 non-null   int64  
 6   duration_minutes               1000 non-null   float64
 7   rating                         1000 non-null   object 
 8   language                       1000 non-null   object 
 9   country_of_origin              1000 non-null   object 
 10  imdb_rating                    1000 non-null   float64
 11  production_budget              1000 non-null   float64
 12  box_office_revenue             1000 non-null   float64

In [41]:
print(df_silver.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   movie_id                       1000 non-null   object 
 1   title                          1000 non-null   object 
 2   content_type                   1000 non-null   object 
 3   genre_primary                  1000 non-null   object 
 4   genre_secondary                1000 non-null   object 
 5   release_year                   1000 non-null   int64  
 6   duration_minutes               1000 non-null   float64
 7   rating                         1000 non-null   object 
 8   language                       1000 non-null   object 
 9   country_of_origin              1000 non-null   object 
 10  imdb_rating                    1000 non-null   float64
 11  production_budget              1000 non-null   float64
 12  box_office_revenue             1000 non-null   float64

## 7. Final Type Formatting and TSQL Readiness

In [42]:
# Converting 'added_to_platform' to datetime format
df_silver['added_to_platform'] = pd.to_datetime(df_silver['added_to_platform'], format='%Y-%m-%d')

df_silver['added_to_platform'] = df_silver['added_to_platform'].dt.strftime('%Y-%m-%d')


In [43]:
bool_columns = ['is_netflix_original','content_warning']

# Converting boolean columns to integer (1/0)
for col in bool_columns:
    df_silver[col] = df_silver[col].astype(int)

In [44]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   movie_id                       1000 non-null   object 
 1   title                          1000 non-null   object 
 2   content_type                   1000 non-null   object 
 3   genre_primary                  1000 non-null   object 
 4   genre_secondary                1000 non-null   object 
 5   release_year                   1000 non-null   int64  
 6   duration_minutes               1000 non-null   float64
 7   rating                         1000 non-null   object 
 8   language                       1000 non-null   object 
 9   country_of_origin              1000 non-null   object 
 10  imdb_rating                    1000 non-null   float64
 11  production_budget              1000 non-null   float64
 12  box_office_revenue             1000 non-null   float64

# 3. Loading data into csv

In [45]:
import csv

In [46]:
output_file = 'netflix_silver_layer_movies.csv'


df_silver.to_csv(
    output_file,
    sep = ',',
    index=False, 
    decimal = '.',
    encoding = 'utf-8',
    quoting = csv.QUOTE_MINIMAL

)